[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/pandera-certified/notebooks/day-06-hypothesis-schema-inference.ipynb#scrollTo=11223344)

---
# Day 6 · Hypothesis Integration and Schema Inference
**certified-journeys / pandera-certified** · Property-based testing and automatic schema discovery

> **Goal for today:** Use Pandera's Hypothesis integration to generate arbitrary valid DataFrames for property-based tests, and use `pa.infer_schema()` to bootstrap a schema from an existing DataFrame.


In [ ]:
%pip install -q "pandera[hypotheses]" hypothesis


## Step 1 · What is property-based testing and why does it matter for data pipelines?

**Unit tests** check a fixed set of inputs you think of in advance. **Property-based tests** let a framework (Hypothesis) generate hundreds of random inputs automatically, searching for counter-examples to a claim you make about your code.

Pandera's `schema.strategy()` returns a **Hypothesis strategy** — an object that knows how to draw random DataFrames that satisfy the schema. Pairing it with `@given` means: *"for any DataFrame that passes this schema, the transformation must also produce a valid result"*.

Key Hypothesis concepts:

| Term | Meaning |
|------|---------|
| `strategy` | Rule for generating random values |
| `@given(...)` | Decorator that runs the test with many generated inputs |
| `settings` | Controls how many examples, max time, etc. |
| Shrinking | Hypothesis minimises a failing example to the simplest form |


In [ ]:
import pandera as pa
import pandas as pd

# Define a simple schema
schema = pa.DataFrameSchema(
    {
        "x": pa.Column(float, pa.Check.gt(0.0)),  # strictly positive float
        "y": pa.Column(int, pa.Check.between(1, 100)),  # integer 1–100
    }
)

# Draw a single sample from the schema's strategy (no @given needed)
from hypothesis import given, settings
import hypothesis.strategies as st

# Use .example() to inspect what strategy() can produce interactively
strategy = schema.strategy(size=5)  # produce DataFrames with 5 rows
sample_df = strategy.example()      # draw one sample
print("Sample DataFrame from schema.strategy():")
print(sample_df)
print(f"\nDtypes: {dict(sample_df.dtypes)}")

# Confirm the sample itself validates
schema.validate(sample_df)
print("\nSample validates against schema: OK")


**What just happened?**

- **`schema.strategy(size=5)`** returns a Hypothesis strategy that generates Pandas DataFrames conforming to all column constraints.
- **`.example()`** draws a single sample — useful for interactive exploration, but should NOT be used inside `@given` tests (it has different semantics).
- Every generated row satisfies `x > 0` and `1 ≤ y ≤ 100` — the schema acts as a generator spec.
- The `size` parameter controls the number of rows per generated DataFrame.


## Step 2 · Write a `@given` test verifying a transformation always produces valid output

The key property to test: *for any valid input, the transformation produces output that also satisfies the output schema*. This is called an **invariant** — it must hold for ALL inputs, not just the ones you happened to write.


In [ ]:
import pandera as pa
import pandas as pd
import numpy as np
from hypothesis import given, settings, HealthCheck

input_schema = pa.DataFrameSchema(
    {
        "x": pa.Column(float, pa.Check.gt(0.0)),
        "y": pa.Column(int, pa.Check.between(1, 100)),
    }
)

output_schema = pa.DataFrameSchema(
    {
        "x": pa.Column(float, pa.Check.gt(0.0)),
        "y": pa.Column(int, pa.Check.between(1, 100)),
        "ratio": pa.Column(float, pa.Check.gt(0.0)),  # x / y must remain positive
    }
)


def add_ratio(df: pd.DataFrame) -> pd.DataFrame:
    """Add ratio = x / y. Since x > 0 and y >= 1, ratio must also be > 0."""
    return df.assign(ratio=df["x"] / df["y"])


@given(input_schema.strategy(size=10))
@settings(
    max_examples=30,                           # run 30 random DataFrames
    suppress_health_check=[HealthCheck.too_slow],
)
def test_add_ratio_invariant(df):
    """For any valid input, the output must satisfy output_schema."""
    result = add_ratio(df)
    output_schema.validate(result)  # raises SchemaError if invariant broken


# Run the property-based test
test_add_ratio_invariant()
print("Property-based test passed: ratio > 0 for all 30 generated inputs.")


**What just happened?**

- **`@given(input_schema.strategy(size=10))`** tells Hypothesis to call `test_add_ratio_invariant` 30 times, each time with a freshly generated DataFrame that satisfies `input_schema`.
- **`output_schema.validate(result)`** inside the test body turns the schema into an assertion — if any generated input produces invalid output, Hypothesis will shrink the example and report the simplest failing case.
- **`suppress_health_check=[HealthCheck.too_slow]`** prevents Hypothesis from aborting on strategy setup time in Colab's slower environment.
- This test would have caught the `feels_like` bug from Day 5 with zero hand-written test cases.


## Step 3 · Adding a custom strategy with `Check.strategy()`

By default, `schema.strategy()` generates values that satisfy the check using Pandera's built-in strategy inference. For checks like `pa.Check.gt(0)`, Pandera automatically produces positive numbers.

For **custom checks** or when you want tighter control over the generated distribution, you can attach a Hypothesis strategy to a `Check` using `.strategy(st.integers(min_value=1))`.


In [ ]:
import pandera as pa
import pandas as pd
import hypothesis.strategies as st
from hypothesis import given, settings, HealthCheck

# Custom strategy: integer between 1 and 1000, skewed toward small values
custom_qty_strategy = st.integers(min_value=1, max_value=1000)

schema_with_custom = pa.DataFrameSchema(
    {
        # Attach a custom strategy to the Check for 'quantity'
        "quantity": pa.Column(
            int,
            pa.Check.ge(1, name="qty_positive"),
            # Override the strategy used for this column's generation
        ),
        "price": pa.Column(
            float,
            pa.Check.gt(0.0, name="price_positive"),
        ),
    }
)

# Register a custom strategy for the 'quantity' column using add_type_extensions
# Simpler approach: build a composite strategy using st.builds / st.fixed_dictionaries
@st.composite
def order_strategy(draw):
    n = draw(st.integers(min_value=1, max_value=10))  # number of rows
    quantities = draw(st.lists(st.integers(min_value=1, max_value=1000), min_size=n, max_size=n))
    prices = draw(st.lists(st.floats(min_value=0.01, max_value=9999.99, allow_nan=False), min_size=n, max_size=n))
    return pd.DataFrame({"quantity": quantities, "price": prices})


@given(order_strategy())
@settings(max_examples=20, suppress_health_check=[HealthCheck.too_slow])
def test_order_invariant(df):
    # All generated DataFrames should pass schema_with_custom
    schema_with_custom.validate(df)
    # And revenue must always be positive
    revenue = (df["quantity"] * df["price"]).sum()
    assert revenue > 0, f"Revenue should be positive, got {revenue}"


test_order_invariant()
print("Custom strategy test passed for 20 generated order DataFrames.")

# Peek at what the strategy generates
sample = order_strategy().example()
print("\nSample from composite strategy:")
print(sample)


**What just happened?**

- **`@st.composite`** builds a custom Hypothesis strategy that draws multiple columns independently and assembles them into a DataFrame.
- The strategy enforces bounds (`min_value=1` for quantity, `min_value=0.01` for price) that align with the schema — generated DataFrames are guaranteed to pass `schema_with_custom`.
- **`allow_nan=False`** on `st.floats` prevents `NaN` values that would fail the `price_positive` check.
- This pattern is useful when column values are not independent (e.g., quantity × price constraints must hold).


## Step 4 · Bootstrapping a schema with `pa.infer_schema()`

When you inherit an existing dataset without a schema, `pa.infer_schema()` inspects the DataFrame and generates a best-effort `DataFrameSchema` — including detected dtypes and basic null/non-null patterns.

Think of it as a **starting point**: the inferred schema won't capture business rules (value ranges, allowed categories), but it gives you the column structure for free.


In [ ]:
import pandera as pa
import pandas as pd

# Simulate a legacy dataset you inherited (no schema exists)
legacy_df = pd.DataFrame(
    {
        "customer_id": [101, 102, 103, 104],
        "name": ["Alice", "Bob", "Carol", "Dave"],
        "age": [28, 35, 42, 19],
        "annual_spend": [1200.50, 890.00, 3400.75, 450.25],
        "premium": [True, False, True, False],
    }
)

# Infer a schema from the existing DataFrame
inferred_schema = pa.infer_schema(legacy_df)
print("Inferred schema:")
print(inferred_schema)


**What just happened?**

- **`pa.infer_schema()`** inspects each column's dtype and null pattern in `legacy_df`.
- The returned `DataFrameSchema` includes a `Column` definition per column with the inferred dtype.
- Checks are minimal — usually just `nullable=False` when no NaN is observed — so you need to add domain-specific `Check` rules manually.
- This is a one-time bootstrapping step; the real value comes from the `to_script()` method that follows.


## Step 5 · `inferred_schema.to_script()` — export to a Python code snippet

`to_script()` converts the inferred schema into a Python source string that you can paste into your codebase and then annotate with additional `Check` rules. This is the recommended workflow:

1. `pa.infer_schema(df)` → bootstrap from real data.
2. `schema.to_script()` → copy the code.
3. Add domain-specific `Check` constraints manually.
4. Commit the schema file — it becomes your contract.


In [ ]:
import pandera as pa

# Export the inferred schema as runnable Python code
script = inferred_schema.to_script()
print("Generated Python schema script:")
print("=" * 60)
print(script)
print("=" * 60)

# You can also write it to a file for version control
schema_path = "/tmp/customer_schema.py"
with open(schema_path, "w") as f:
    f.write(script)
print(f"\nSchema saved to {schema_path}")

# Demonstrate round-trip: validate the original DataFrame with the inferred schema
inferred_schema.validate(legacy_df)
print("Round-trip validation passed: legacy_df satisfies the inferred schema.")


**What just happened?**

- **`to_script()`** serializes the `DataFrameSchema` to a Python source string with proper imports.
- The output is a complete, runnable module — `exec`-able or paste-able into a schema definition file.
- The round-trip validation confirms the inferred schema correctly describes the source DataFrame.
- **Next step in practice:** open the generated file, add `pa.Check.ge(1)` for `age`, `pa.Check.gt(0)` for `annual_spend`, etc.


## Step 6 · Lazy validation — collecting all failures before raising

Reviewed briefly in Day 4; here we go deeper with a focus on **how to build a validation report** from `SchemaErrors.failure_cases`.

The `failure_cases` DataFrame has these columns:

| Column | Meaning |
|--------|---------|
| `schema_context` | Type of violation (`Column`, `DataFrameSchema`) |
| `column` | Column name where failure occurred |
| `check` | Check name that failed |
| `check_number` | Numeric ID of the check |
| `failure_case` | The actual violating value |
| `index` | Row index of the failure |


In [ ]:
import pandera as pa
import pandas as pd

strict_schema = pa.DataFrameSchema(
    {
        "customer_id": pa.Column(int, pa.Check.ge(100, name="id_ge_100")),
        "age": pa.Column(int, pa.Check.between(0, 120, name="age_range")),
        "annual_spend": pa.Column(float, pa.Check.gt(0.0, name="spend_positive")),
        "premium": pa.Column(bool),
    }
)

# DataFrame with multiple violations across different columns
dirty_df = pd.DataFrame(
    {
        "customer_id": [50, 102, 99, 105],    # rows 0 and 2 fail ge(100)
        "age": [28, -5, 200, 40],              # rows 1 and 2 fail between(0, 120)
        "annual_spend": [500.0, -10.0, 800.0, 0.0],  # rows 1 and 3 fail gt(0)
        "premium": [True, False, True, False],
    }
)

try:
    strict_schema.validate(dirty_df, lazy=True)
except pa.errors.SchemaErrors as errs:
    report = errs.failure_cases
    print("Validation report (all failures):")
    print(report.to_string())
    print(f"\nTotal violations : {len(report)}")
    print(f"Affected columns : {report['column'].unique().tolist()}")
    print(f"Affected rows    : {report['index'].dropna().astype(int).unique().tolist()}")


**What just happened?**

- **`lazy=True`** runs all checks before raising, so the full `failure_cases` report contains all 5 violations across 3 columns.
- The **`column`** and **`check`** fields let you group failures by rule or by column for dashboard-friendly reporting.
- **`errs.data`** (not shown) gives you the original (unchanged) input DataFrame for side-by-side comparison.
- In production pipelines, route `failure_cases.to_dict(orient="records")` to a monitoring sink (Datadog, CloudWatch, etc.).


In [ ]:
# Challenge: Infer + test + report
#
# 1. Create a sample DataFrame representing product inventory:
#    - product_id: int
#    - name: str
#    - stock: int (should be >= 0)
#    - price: float (should be > 0)
#    - active: bool
#
# 2. Use pa.infer_schema() to bootstrap a schema, then call to_script() and print it.
#
# 3. Manually add Check.ge(0) to stock and Check.gt(0) to price in the inferred schema.
#
# 4. Write a @given(schema.strategy(size=5)) property test that asserts:
#    - All generated DataFrames have stock >= 0 and price > 0
#
# 5. Create a dirty DataFrame with 3 violations and validate with lazy=True,
#    then print a summary: total violations and affected columns.

# Your solution here


---
## Day 6 key concepts recap

| Concept | What to remember |
|---|---|
| `schema.strategy(size=N)` | Hypothesis strategy that generates valid DataFrames with N rows |
| `.example()` | Draws one sample for interactive exploration (don't use inside `@given`) |
| `@given(schema.strategy())` | Runs the test with many randomly generated valid inputs |
| `@st.composite` | Build a custom multi-column Hypothesis strategy |
| `pa.infer_schema(df)` | Bootstrap a `DataFrameSchema` from an existing DataFrame |
| `schema.to_script()` | Export the schema to runnable Python source code |
| `lazy=True` | Collect ALL violations; raises `pa.errors.SchemaErrors` (plural) |
| `failure_cases` columns | `schema_context`, `column`, `check`, `failure_case`, `index` |

> **Tip:** Run `@given` tests in CI with `max_examples=200` for thorough coverage; use a lower count locally (`max_examples=20`) to keep feedback loops fast.

---
## What's next
**Day 7** → Capstone: build a fully validated three-stage production pipeline with `DataFrameModel` schemas, decorator guards, lazy validation reports, and complete Hypothesis property tests.

Mark Day 6 complete in your [tracker](../index.html).
